# Predict Tags on 10 Test Dataset Samples
### Ground Truth vs. Old `model.plan` (TensorRT) vs. Multi-Class RF-DETR

This notebook evaluates **10 test dataset images** with side-by-side comparisons:
1. **Ground Truth** — Actual verified bounding boxes and tags from `_annotations.coco.json`.
2. **Old Model (`model.plan`)** — Universal TensorRT engine (DBNet text detector / YOLO formats).
3. **Multi-Class RF-DETR** — Fine-tuned PyTorch model (`blue_aisle`, `blue_bay`, `location_tag`).

All 10 samples are rendered as a 3-panel comparison with bold bounding boxes and confidence score badges.

In [ ]:
# 0. Auto-Install Dependencies (ONNX Runtime, TensorRT & CUDA Packages)
import sys
import subprocess
import importlib
import os
import re

print('=' * 75)
print('CHECKING & INSTALLING REQUIRED PACKAGES (ONNX Runtime, TensorRT & CUDA)')
print('=' * 75)

def check_and_install_packages():
    # 1. Install ONNX & ONNX Runtime (Universal, no TRT version mismatch issues)
    try:
        import onnxruntime as ort
        print(f'✓ ONNX Runtime already installed: version {ort.__version__}')
    except ImportError:
        print('ONNX Runtime not found. Installing onnxruntime and onnx...')
        try:
            import torch
            has_cuda = torch.cuda.is_available()
        except Exception:
            has_cuda = False
        ort_pkg = 'onnxruntime-gpu' if has_cuda else 'onnxruntime'
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', ort_pkg, 'onnx'])
            importlib.invalidate_caches()
            import onnxruntime as ort
            print(f'✓ Successfully installed ONNX Runtime: version {ort.__version__}')
        except Exception as e:
            print(f'Installing standard onnxruntime fallback: {e}')
            try:
                subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', 'onnxruntime', 'onnx'])
                importlib.invalidate_caches()
                import onnxruntime as ort
                print(f'✓ Successfully installed ONNX Runtime: version {ort.__version__}')
            except Exception as e2:
                print(f'ONNX install note: {e2}')

    # 2. Install TensorRT & CUDA packages
    try:
        import tensorrt as trt
        print(f'✓ TensorRT already installed: version {trt.__version__}')
    except ImportError:
        print('TensorRT not found. Detecting CUDA version and installing TensorRT...')
        cuda_major = None
        try:
            import torch
            if torch.cuda.is_available() and torch.version.cuda:
                cuda_major = torch.version.cuda.split('.')[0]
                print(f'Detected CUDA version: {torch.version.cuda} (Major: {cuda_major})')
        except Exception:
            pass

        if cuda_major == '12':
            pkgs = ['tensorrt-cu12', 'cuda-python']
        elif cuda_major == '11':
            pkgs = ['tensorrt-cu11', 'cuda-python']
        else:
            pkgs = ['tensorrt', 'cuda-python']

        print(f'Attempting installation: {pkgs} ...')
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U'] + pkgs)
            importlib.invalidate_caches()
            import tensorrt as trt
            print(f'✓ Successfully installed TensorRT: version {trt.__version__}')
        except Exception as e:
            print(f'Trying universal "tensorrt" package...')
            try:
                subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', 'tensorrt', 'cuda-python'])
                importlib.invalidate_caches()
                import tensorrt as trt
                print(f'✓ Successfully installed TensorRT: version {trt.__version__}')
            except Exception as e2:
                print(f'TensorRT auto-install note: {e2}')

check_and_install_packages()

# To iteratively test and install TensorRT versions (starting from 8.4) until your .plan loads:
# !python install_until_plan_loads.py


In [ ]:
# 1. Imports & Environment Setup
import os
import re
import json
import cv2
import torch
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
from torchvision.ops import batched_nms

# TensorRT check
try:
    import tensorrt as trt
    HAS_TRT = True
except ImportError:
    import importlib
    importlib.invalidate_caches()
    try:
        import tensorrt as trt
        HAS_TRT = True
    except ImportError:
        HAS_TRT = False

# ONNX Runtime check
try:
    import onnxruntime as ort
    HAS_ORT = True
except ImportError:
    import importlib
    importlib.invalidate_caches()
    try:
        import onnxruntime as ort
        HAS_ORT = True
    except ImportError:
        HAS_ORT = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trt_info = f"Version {trt.__version__}" if HAS_TRT else "Not Available"
ort_info = f"Version {ort.__version__}" if HAS_ORT else "Not Available"
print(f"Inference Device: {device}")
print(f"TensorRT:         {trt_info}")
print(f"ONNX Runtime:     {ort_info}")


In [ ]:
# 2. Load 10 Test Dataset Samples WITH Ground Truth Annotations
NUM_SAMPLES = 10

TEST_DIR = "multi_class_train_rfdetr/dataset_full_data/test"
if not os.path.exists(TEST_DIR):
    TEST_DIR = "multi_class_train_rfdetr/dataset_sample_1000/test"

IMAGES_DIR = "multi_class_train_rfdetr/images"
ANN_FILE = os.path.join(TEST_DIR, "_annotations.coco.json")

CLASSES = ["blue_aisle", "blue_bay", "location_tag"]
test_samples = []

if os.path.exists(ANN_FILE):
    with open(ANN_FILE, "r") as f:
        coco = json.load(f)
        
    CLASSES = [c["name"] for c in sorted(coco.get("categories", []), key=lambda x: x["id"])]
    cat_id_to_name = {c["id"]: c["name"] for c in coco.get("categories", [])}
    
    # Group annotations by image_id
    anns_by_img = {}
    for ann in coco.get("annotations", []):
        anns_by_img.setdefault(ann["image_id"], []).append(ann)
        
    # Collect images that have ground truth bounding boxes first
    images_with_gt = []
    images_without_gt = []
    
    for im in coco.get("images", []):
        img_id = im["id"]
        fname = im["file_name"]
        p = os.path.join(IMAGES_DIR, fname)
        if not os.path.exists(p):
            alt = os.path.join(TEST_DIR, fname)
            if os.path.exists(alt): p = alt
            
        img_anns = anns_by_img.get(img_id, [])
        boxes, labels = [], []
        for a in img_anns:
            bx, by, bw, bh = a["bbox"]
            if bw > 0 and bh > 0:
                boxes.append([bx, by, bx + bw, by + bh])
                labels.append(cat_id_to_name.get(a["category_id"], "location_tag"))
                
        item = {"path": p, "file_name": fname, "gt_boxes": boxes, "gt_labels": labels}
        if len(boxes) > 0:
            images_with_gt.append(item)
        else:
            images_without_gt.append(item)
            
    test_samples = (images_with_gt + images_without_gt)[:NUM_SAMPLES]
else:
    test_samples = [{"path": f"test_{i}.jpg", "file_name": f"test_{i}.jpg", "gt_boxes": [], "gt_labels": []} for i in range(1, NUM_SAMPLES + 1)]

print(f"Target Categories ({len(CLASSES)}): {CLASSES}")
print(f"Selected {len(test_samples)} Test Samples for evaluation:")
for idx, s in enumerate(test_samples, 1):
    print(f"  {idx:2d}. {s['file_name']:<30} -> {len(s['gt_boxes'])} Ground Truth tag(s) (exists: {os.path.exists(s['path'])})")


In [ ]:
# 3. Load Models (Multi-Class RF-DETR & Old model.plan - Dimensions Loaded from Model Only)

# --- 3A. Multi-Class RF-DETR Model Loading ---
from rfdetr import RFDETRBase
from rfdetr.models.lwdetr import LWDETR

# Prevent unwanted background downloads
RFDETRBase.maybe_download_pretrain_weights = lambda self: None
RFDETRBase.load_pretrain_weights = lambda self: None

# Safe weight loader to match layer tensor shapes
def _safe_load(self, state_dict, strict=True):
    cur = self.state_dict()
    filtered = {k.replace("model.", "").replace("module.", ""): v for k, v in state_dict.items()}
    filtered = {k: v for k, v in filtered.items() if k in cur and cur[k].shape == v.shape}
    return torch.nn.Module.load_state_dict(self, filtered, strict=False)
LWDETR.load_state_dict = _safe_load

def reinitialize_heads(lwdetr, num_classes):
    """Properly reinitializes class_embed and transformer.enc_out_class_embed (ModuleList)."""
    dev = next(lwdetr.parameters()).device
    if hasattr(lwdetr, "class_embed"):
        in_f = lwdetr.class_embed.in_features
        lwdetr.class_embed = torch.nn.Linear(in_f, num_classes).to(dev)
    if hasattr(lwdetr, "transformer") and hasattr(lwdetr.transformer, "enc_out_class_embed"):
        enc_heads = lwdetr.transformer.enc_out_class_embed
        if isinstance(enc_heads, torch.nn.ModuleList):
            for i in range(len(enc_heads)):
                in_f = enc_heads[i].in_features
                enc_heads[i] = torch.nn.Linear(in_f, num_classes).to(dev)
        elif isinstance(enc_heads, torch.nn.Linear):
            in_f = enc_heads.in_features
            lwdetr.transformer.enc_out_class_embed = torch.nn.Linear(in_f, num_classes).to(dev)

# Search candidate checkpoint paths
CANDIDATE_CKPTS = [
    "multi_class_train_rfdetr/model/best_model_full_data.pth",
    "multi_class_train_rfdetr/model/best_model_sample_1000.pth",
    "multi_class_train_rfdetr/runs/rf_detr_full_data/checkpoints/best_loss.pth",
    "multi_class_train_rfdetr/runs/rf_detr_sample_1000/checkpoints/best_loss.pth",
    "location_tag_single_class_train_rfdetr/model/best_model_full_data.pth",
    "location_tag_single_class_train_rfdetr/model/best_model_sample_1000.pth",
    "best_model_full_data.pth",
    "best_model.pth"
]
rf_ckpt = None
for cp in CANDIDATE_CKPTS:
    if os.path.exists(cp) and os.path.getsize(cp) > 1024 * 1024:
        rf_ckpt = cp
        break
if rf_ckpt is None:
    rf_ckpt = CANDIDATE_CKPTS[0]

wrapper = RFDETRBase(num_classes=len(CLASSES), resolution=1008, pretrain_weights=None)
if hasattr(wrapper, "model") and hasattr(wrapper.model, "model") and hasattr(wrapper.model.model, "load_state_dict"):
    rfdetr = wrapper.model.model
elif hasattr(wrapper, "model") and hasattr(wrapper.model, "load_state_dict"):
    rfdetr = wrapper.model
else:
    rfdetr = wrapper

# Load weights and adapt detection heads to checkpoint class count
if os.path.exists(rf_ckpt):
    ckpt = torch.load(rf_ckpt, map_location="cpu", weights_only=False)
    state = ckpt.get("model", ckpt)
    for k in ["class_embed.weight", "model.class_embed.weight", "module.class_embed.weight"]:
        if k in state:
            ckpt_n_classes = state[k].shape[0]
            if hasattr(rfdetr, "class_embed") and rfdetr.class_embed.out_features != ckpt_n_classes:
                reinitialize_heads(rfdetr, ckpt_n_classes)
                print(f"Reinitialized detection heads for {ckpt_n_classes} classes.")
            break
            
    rfdetr.load_state_dict(state, strict=False)
    print(f"Loaded RF-DETR checkpoint from: {rf_ckpt} ({os.path.getsize(rf_ckpt)/1e6:.1f} MB)")
else:
    print(f"WARNING: RF-DETR checkpoint not found at: {rf_ckpt}")
rfdetr.to(device).eval()

# --- 3B. Universal Old Model Runner (Supports TensorRT .plan AND ONNX Runtime .onnx) ---

def inspect_plan_version(plan_path: str) -> str:
    """Inspects a .plan file to detect the TensorRT version embedded in its header."""
    if not os.path.exists(plan_path):
        return "File not found"
    try:
        with open(plan_path, "rb") as f:
            chunk = f.read(1024)
        matches = re.findall(rb"(\d{1,2}\.\d{1,2}\.\d{1,2}(?:\.\d{1,2})?)", chunk)
        if matches:
            for m in matches:
                v = m.decode("ascii")
                if v.startswith(("8.", "9.", "10.")):
                    return v
        return "Unknown"
    except Exception:
        return "Unknown"


class TensorRTRunner:
    """Runs inference on a TensorRT .plan engine."""
    def __init__(self, plan_path: str, device: torch.device):
        self.plan_path = plan_path
        self.device = device
        self.is_ready = False
        self.inputs, self.outputs = [], []
        self.in_h, self.in_w = 640, 640
        
        if not HAS_TRT:
            print("[TensorRT Runner] Notice: tensorrt library not available.")
            return
        if not os.path.exists(plan_path):
            print(f"[TensorRT Runner] Notice: Plan file not found at '{plan_path}'.")
            return
            
        try:
            class TRTLogger(trt.ILogger):
                def __init__(self):
                    super().__init__()
                def log(self, severity, msg):
                    if severity in (trt.ILogger.Severity.INTERNAL_ERROR, trt.ILogger.Severity.ERROR):
                        print(f"   [TensorRT ERROR]: {msg}")
                    elif severity == trt.ILogger.Severity.WARNING:
                        print(f"   [TensorRT WARNING]: {msg}")

            trt_logger = TRTLogger()
            with open(plan_path, "rb") as f:
                plan_data = f.read()
            print(f"[TensorRT Runner] Read engine file '{plan_path}' ({len(plan_data)/1e6:.1f} MB)")
            
            with trt.Runtime(trt_logger) as runtime:
                self.engine = runtime.deserialize_cuda_engine(plan_data)
            if self.engine is None:
                print(f"[TensorRT Runner] Deserialization returned None for '{plan_path}'.")
                plan_ver = inspect_plan_version(plan_path)
                print(f"   -> Required Engine TRT Version: {plan_ver} (Installed: {trt.__version__})")
                return
                
            self.context = self.engine.create_execution_context()
            self._inspect_io()
            self._load_dimensions_from_model()
            self.is_ready = True
            print(f"[TensorRT Runner] Engine loaded successfully!")
            print(f"[TensorRT Runner] Input Dimensions Loaded Directly from Model: {self.in_w}x{self.in_h}")
        except Exception as e:
            print(f"[TensorRT Runner] Error initializing engine: {e}")

    def _inspect_io(self):
        if hasattr(self.engine, "num_io_tensors"):
            for i in range(self.engine.num_io_tensors):
                name = self.engine.get_tensor_name(i)
                mode = self.engine.get_tensor_mode(name)
                shape = list(self.engine.get_tensor_shape(name))
                meta = {"name": name, "shape": shape}
                (self.inputs if mode == trt.TensorIOMode.INPUT else self.outputs).append(meta)
        else:
            for i in range(self.engine.num_bindings):
                name = self.engine.get_binding_name(i)
                is_in = self.engine.binding_is_input(i)
                shape = list(self.engine.get_binding_shape(i))
                meta = {"name": name, "shape": shape}
                (self.inputs if is_in else self.outputs).append(meta)

    def _load_dimensions_from_model(self):
        """Extracts input dimensions directly and exclusively from the model.plan engine."""
        if not self.inputs:
            return
        in_meta = self.inputs[0]
        in_shape = list(in_meta["shape"])
        h, w = in_shape[-2], in_shape[-1]
        
        if h <= 0 or w <= 0:
            try:
                if hasattr(self.engine, "get_tensor_profile_shape"):
                    profiles = self.engine.get_tensor_profile_shape(in_meta["name"], 0)
                    opt_shape = profiles[1]
                    h, w = opt_shape[-2], opt_shape[-1]
                elif hasattr(self.engine, "get_profile_shape"):
                    profiles = self.engine.get_profile_shape(0, in_meta["name"])
                    opt_shape = profiles[1]
                    h, w = opt_shape[-2], opt_shape[-1]
            except Exception:
                h, w = 640, 640
                
        self.in_h = int(h) if h > 0 else 640
        self.in_w = int(w) if w > 0 else 640

    def infer(self, input_tensor: torch.Tensor):
        if not self.is_ready:
            return []
        input_tensor = input_tensor.contiguous().to(self.device)
        output_tensors = []
        
        # Method 1: TensorRT 8.5+ / 10.x API
        if hasattr(self.context, "set_tensor_address") and hasattr(self.context, "execute_async_v3"):
            try:
                self.context.set_tensor_address(self.inputs[0]["name"], input_tensor.data_ptr())
                for out_m in self.outputs:
                    shape = [input_tensor.shape[0] if s < 0 and idx == 0 else (abs(s) if s < 0 else s) for idx, s in enumerate(out_m["shape"])]
                    out_t = torch.empty(shape, dtype=torch.float32, device=self.device)
                    self.context.set_tensor_address(out_m["name"], out_t.data_ptr())
                    output_tensors.append(out_t)
                self.context.execute_async_v3(torch.cuda.current_stream().cuda_stream)
                torch.cuda.synchronize()
                return output_tensors
            except Exception:
                output_tensors = []
                
        # Method 2: TensorRT Legacy bindings API (v8.0 - v8.4)
        try:
            bindings = [0] * (len(self.inputs) + len(self.outputs))
            bindings[0] = input_tensor.data_ptr()
            output_tensors = []
            for idx, out_m in enumerate(self.outputs, start=len(self.inputs)):
                shape = [abs(s) if s != 0 else 1 for s in out_m["shape"]]
                out_t = torch.empty(shape, dtype=torch.float32, device=self.device)
                bindings[idx] = out_t.data_ptr()
                output_tensors.append(out_t)
            if hasattr(self.context, "execute_async_v2"):
                self.context.execute_async_v2(bindings, torch.cuda.current_stream().cuda_stream)
            elif hasattr(self.context, "execute_v2"):
                self.context.execute_v2(bindings)
            torch.cuda.synchronize()
            return output_tensors
        except Exception as e:
            print(f"[TensorRT infer warning]: {e}")
            return []


class ONNXRunner:
    """Runs inference on an ONNX model via ONNX Runtime (CUDA / CPU)."""
    def __init__(self, onnx_path: str, device: torch.device):
        self.onnx_path = onnx_path
        self.device = device
        self.is_ready = False
        self.session = None
        self.in_h, self.in_w = 640, 640
        self.input_name = None
        self.output_names = []
        
        if not HAS_ORT:
            print("[ONNX Runner] onnxruntime not installed.")
            return
        if not os.path.exists(onnx_path):
            print(f"[ONNX Runner] ONNX file not found at: '{onnx_path}'")
            return
            
        try:
            providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if device.type == "cuda" else ["CPUExecutionProvider"]
            self.session = ort.InferenceSession(onnx_path, providers=providers)
            inp = self.session.get_inputs()[0]
            self.input_name = inp.name
            shape = inp.shape
            h = shape[-2] if (len(shape) >= 2 and isinstance(shape[-2], int) and shape[-2] > 0) else 640
            w = shape[-1] if (len(shape) >= 1 and isinstance(shape[-1], int) and shape[-1] > 0) else 640
            self.in_h, self.in_w = int(h), int(w)
            self.output_names = [out.name for out in self.session.get_outputs()]
            self.is_ready = True
            active_p = self.session.get_providers()[0]
            print(f"[ONNX Runner] Model loaded successfully from: {onnx_path}")
            print(f"[ONNX Runner] Input: '{self.input_name}' | Resolution: {self.in_w}x{self.in_h} | Provider: {active_p}")
        except Exception as e:
            print(f"[ONNX Runner] Error initializing session: {e}")

    def infer(self, input_tensor: torch.Tensor):
        if not self.is_ready or self.session is None:
            return []
        np_inp = input_tensor.detach().cpu().numpy()
        raw_outs = self.session.run(self.output_names, {self.input_name: np_inp})
        return [torch.from_numpy(o).to(self.device) for o in raw_outs]


def convert_onnx_to_plan(onnx_path: str, plan_path: str, fp16: bool = True) -> bool:
    """Compiles an ONNX model into a fresh TensorRT .plan engine matching current TRT version."""
    if not HAS_TRT:
        print("TensorRT not available for building engine.")
        return False
    if not os.path.exists(onnx_path):
        print(f"Source ONNX file not found: {onnx_path}")
        return False
    try:
        logger = trt.Logger(trt.Logger.INFO)
        builder = trt.Builder(logger)
        network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
        parser = trt.OnnxParser(network, logger)
        with open(onnx_path, "rb") as f:
            if not parser.parse(f.read()):
                for error in range(parser.num_errors):
                    print(f"ONNX Parser Error: {parser.get_error(error)}")
                return False
        config = builder.create_builder_config()
        if fp16 and builder.platform_has_fast_fp16:
            config.set_flag(trt.BuilderFlag.FP16)
        if hasattr(config, "set_memory_pool_limit"):
            config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)
        print(f"Compiling '{onnx_path}' into TensorRT engine matching installed version ({trt.__version__})...")
        serialized_engine = builder.build_serialized_network(network, config)
        if serialized_engine is None:
            print("Failed to build serialized engine.")
            return False
        with open(plan_path, "wb") as f:
            f.write(serialized_engine)
        print(f"✓ Successfully built and saved: '{plan_path}'")
        return True
    except Exception as e:
        print(f"Error compiling ONNX to TensorRT plan: {e}")
        return False


class UniversalOldModelRunner:
    """Universal Runner that loads TensorRT .plan and gracefully falls back to .onnx if version mismatch occurs."""
    def __init__(self, plan_path: str, onnx_path: str, device: torch.device):
        self.device = device
        self.active_runner = None
        self.backend = "None"
        self.model_file = None
        self.is_ready = False
        self.in_h, self.in_w = 640, 640
        
        # 1. Try TensorRT runner if .plan file exists
        if os.path.exists(plan_path):
            print(f"[Universal Runner] Trying TensorRT engine: {plan_path}")
            trt_run = TensorRTRunner(plan_path, device)
            if trt_run.is_ready:
                self.active_runner = trt_run
                self.backend = "TensorRT (.plan)"
                self.model_file = plan_path
                self.is_ready = True
                self.in_h, self.in_w = trt_run.in_h, trt_run.in_w
                return
            else:
                print(f"[Universal Runner] TensorRT engine failed to load.")
                plan_ver = inspect_plan_version(plan_path)
                print(f"   -> Required Engine TRT Version: {plan_ver}")
                if HAS_TRT:
                    print(f"   -> Current Installed TRT Version: {trt.__version__}")
                print("   -> Checking for ONNX model fallback...")

        # 2. Try ONNX runner if .onnx file exists
        if onnx_path and os.path.exists(onnx_path):
            print(f"[Universal Runner] Loading ONNX model: {onnx_path}")
            onnx_run = ONNXRunner(onnx_path, device)
            if onnx_run.is_ready:
                self.active_runner = onnx_run
                self.backend = "ONNX Runtime (.onnx)"
                self.model_file = onnx_path
                self.is_ready = True
                self.in_h, self.in_w = onnx_run.in_h, onnx_run.in_w
                return
                
        print(f"[Universal Runner] Old model could not be loaded as .plan or .onnx.")

    def infer(self, input_tensor: torch.Tensor):
        if self.active_runner is not None:
            return self.active_runner.infer(input_tensor)
        return []


# Auto-discover .plan and .onnx models
CANDIDATE_PLANS = [
    "location_tag_text_dat/1/model.plan",
    "location_tag_text_det/1/model.plan",
    "../location_tag_text_dat/1/model.plan",
    "../location_tag_text_det/1/model.plan",
    "location_tag_text_dat/model.plan",
    "location_tag_text_det/model.plan",
    "1/model.plan",
    "model.plan",
    "../model.plan",
    "location_tag_text_det.engine",
    "location_tag_text_det.plan",
]
PLAN_MODEL_PATH = "location_tag_text_dat/1/model.plan"
for cp in CANDIDATE_PLANS:
    if os.path.exists(cp) and os.path.getsize(cp) > 1024:
        PLAN_MODEL_PATH = cp
        break

CANDIDATE_ONNX = [
    "location_tag_text_det.onnx",
    "location_tag_text_dat/model.onnx",
    "location_tag_text_dat/1/model.onnx",
    "location_tag_text_det/model.onnx",
    "location_tag_text_det/1/model.onnx",
    "../location_tag_text_det.onnx",
    "../location_tag_text_dat/model.onnx",
    "../location_tag_text_dat/1/model.onnx",
    "model.onnx",
    "../model.onnx",
    "location_tag.onnx",
    "models/model.onnx"
]
ONNX_MODEL_PATH = None
for co in CANDIDATE_ONNX:
    if os.path.exists(co) and os.path.getsize(co) > 1024:
        ONNX_MODEL_PATH = co
        break

# Recursive scan if not found in candidates
if not os.path.exists(PLAN_MODEL_PATH) or ONNX_MODEL_PATH is None:
    for root_dir in [".", ".."]:
        if os.path.exists(root_dir):
            for root, dirs, files in os.walk(root_dir):
                dirs[:] = [d for d in dirs if not d.startswith(".") and d not in ("venv", "node_modules", ".git")]
                for f in files:
                    if not os.path.exists(PLAN_MODEL_PATH) and (f.endswith(".plan") or f.endswith(".engine")):
                        p = os.path.join(root, f)
                        if os.path.getsize(p) > 1024: PLAN_MODEL_PATH = p
                    if ONNX_MODEL_PATH is None and f.endswith(".onnx"):
                        p = os.path.join(root, f)
                        if os.path.getsize(p) > 1024: ONNX_MODEL_PATH = p

trt_runner = UniversalOldModelRunner(PLAN_MODEL_PATH, ONNX_MODEL_PATH, device)
plan_in_w = trt_runner.in_w
plan_in_h = trt_runner.in_h

print("=" * 75)
print("OLD MODEL STATUS:")
print(f"   - Selected Backend: {trt_runner.backend}")
print(f"   - Loaded File:      {trt_runner.model_file}")
print(f"   - Plan Path:        {PLAN_MODEL_PATH} (Exists: {os.path.exists(PLAN_MODEL_PATH)})")
print(f"   - ONNX Path:        {ONNX_MODEL_PATH} (Exists: {ONNX_MODEL_PATH is not None and os.path.exists(ONNX_MODEL_PATH)})")
print(f"   - Model Dimension:  {plan_in_w}x{plan_in_h}")
print(f"   - Runner Ready:     {trt_runner.is_ready}")
print("=" * 75)


In [ ]:
# 4. Prediction Functions (with Diagnostics & Low Threshold)
CONF_THRESH = 0.15
IOU_THRESH = 0.50

def box_cxcywh_to_xyxy(boxes: torch.Tensor) -> torch.Tensor:
    cx, cy, w, h = boxes.unbind(-1)
    return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dim=-1)

def predict_tags_rfdetr(img_path, conf_thresh=CONF_THRESH):
    """Predicts tags using Multi-Class RF-DETR."""
    if not os.path.exists(img_path):
        return []
    
    pil_img = Image.open(img_path).convert("RGB")
    orig_w, orig_h = pil_img.size
    
    # Resize to 1008x1008 and normalize
    resized = pil_img.resize((1008, 1008), Image.BILINEAR)
    x = TF.to_tensor(resized).unsqueeze(0)
    x = TF.normalize(x, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]).to(device)
    
    with torch.no_grad():
        with torch.amp.autocast(device_type="cuda", enabled=torch.cuda.is_available()):
            out = rfdetr(x)
        
    logits = out["pred_logits"][0]
    boxes = out["pred_boxes"][0]
    
    scores, classes = logits.sigmoid().max(-1)
    keep = scores > conf_thresh
    
    preds = []
    for s, c, (cx, cy, w, h) in zip(scores[keep], classes[keep], boxes[keep]):
        x1 = float(max(0, (cx - w / 2) * orig_w))
        y1 = float(max(0, (cy - h / 2) * orig_h))
        x2 = float(min(orig_w, (cx + w / 2) * orig_w))
        y2 = float(min(orig_h, (cy + h / 2) * orig_h))
        cid = int(c)
        tag_name = CLASSES[cid] if cid < len(CLASSES) else f"class_{cid}"
        preds.append({"tag": tag_name, "class_id": cid, "score": round(float(s), 2), "box": [round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)]})
        
    if len(preds) == 0 and len(scores) > 0:
        max_sc = scores.max().item()
        if max_sc > 0.03:
            top_k = scores.topk(min(3, len(scores)))
            for s, idx in zip(top_k.values, top_k.indices):
                if s > 0.05:
                    cx, cy, w, h = boxes[idx]
                    cid = int(classes[idx])
                    tag_name = CLASSES[cid] if cid < len(CLASSES) else f"class_{cid}"
                    preds.append({"tag": tag_name, "class_id": cid, "score": round(float(s), 2), "box": [round(float((cx-w/2)*orig_w), 1), round(float((cy-h/2)*orig_h), 1), round(float((cx+w/2)*orig_w), 1), round(float((cy+h/2)*orig_h), 1)]})
    return preds


def predict_tags_plan(img_path, conf_thresh=CONF_THRESH):
    """Predicts tags using old model.plan. All tags labeled as 'location_tag' by default."""
    if not trt_runner.is_ready or not os.path.exists(img_path):
        return []
        
    cv_img = cv2.imread(img_path)
    if cv_img is None:
        return []
    orig_h, orig_w = cv_img.shape[:2]
    
    # Preprocess image to engine input shape
    resized = cv2.resize(cv_img, (plan_in_w, plan_in_h))
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    inp_trt = torch.from_numpy(rgb).permute(2, 0, 1).float().div(255.0).unsqueeze(0).to(device)
    
    with torch.no_grad():
        raw_outs = trt_runner.infer(inp_trt)
    if not raw_outs:
        return []
        
    out = raw_outs[0]
    preds = []
    
    # Decoder A: DBNet / Text Detection (Probability map [1, 1, H, W] or [1, H, W])
    if (out.ndim == 4 and out.shape[1] in (1, 2)) or (out.ndim == 3 and out.shape[0] in (1, 2) and out.shape[1] > 10):
        prob_map = out[0, 0] if out.ndim == 4 else out[0]
        # Apply sigmoid if values are raw logits (min < 0 or max > 1)
        if prob_map.min() < 0.0 or prob_map.max() > 1.0:
            prob_map = prob_map.sigmoid()
            
        p_max = float(prob_map.max().item())
        det_thresh = min(conf_thresh, max(0.12, p_max * 0.35))
        mask = (prob_map > det_thresh).cpu().numpy().astype(np.uint8)
        contours, _ = cv2.findContours(mask, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            bx, by, bw, bh = cv2.boundingRect(cnt)
            if bw >= 2 and bh >= 2:
                sub_mask = mask[by:by+bh, bx:bx+bw]
                box_score = float(prob_map[by:by+bh, bx:bx+bw].mean().item()) if sub_mask.size > 0 else 0.60
                if box_score >= det_thresh:
                    x1 = float(bx * (orig_w / plan_in_w))
                    y1 = float(by * (orig_h / plan_in_h))
                    x2 = float((bx + bw) * (orig_w / plan_in_w))
                    y2 = float((by + bh) * (orig_h / plan_in_h))
                    # All tags are labeled 'location_tag' by default for the old model
                    preds.append({"tag": "location_tag", "class_id": 0, "score": round(box_score, 2), "box": [round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)]})
                    
    # Decoder B: YOLO / Object Detection Format
    elif out.ndim in (2, 3):
        p = out[0] if out.ndim == 3 else out
        if p.shape[0] < p.shape[1] and p.shape[0] <= 100:
            p = p.transpose(0, 1)
        if p.shape[1] >= 5:
            if p.shape[1] == 5:
                scs = p[:, 4].sigmoid() if (p[:, 4].max() > 1.0 or p[:, 4].min() < 0) else p[:, 4]
                lbs = torch.zeros(len(p), dtype=torch.long, device=p.device)
            else:
                scores_all = p[:, 4:].sigmoid() if (p[:, 4:].max() > 1.0 or p[:, 4:].min() < 0) else p[:, 4:]
                scs, lbs = scores_all.max(-1)
                
            keep = scs > conf_thresh
            if keep.sum() > 0:
                b_xyxy = box_cxcywh_to_xyxy(p[keep, :4])
                nms_k = batched_nms(b_xyxy, scs[keep], lbs[keep], IOU_THRESH)
                kept_boxes = b_xyxy[nms_k]
                
                # Handle normalized (<=1.5) vs pixel (>1.5) box coordinates
                if kept_boxes.max() <= 1.5:
                    scaled = kept_boxes * torch.tensor([orig_w, orig_h, orig_w, orig_h], device=b_xyxy.device)
                else:
                    scaled = kept_boxes * torch.tensor([orig_w/plan_in_w, orig_h/plan_in_h, orig_w/plan_in_w, orig_h/plan_in_h], device=b_xyxy.device)
                    
                for box, score in zip(scaled.cpu().tolist(), scs[keep][nms_k].cpu().tolist()):
                    # All tags are labeled 'location_tag' by default for the old model
                    preds.append({"tag": "location_tag", "class_id": 0, "score": round(float(score), 2), "box": [round(x, 1) for x in box]})
                    
    return preds


In [ ]:
# 5. Run Prediction on the 10 Test Images & Print Detected Tags
for i, sample in enumerate(test_samples, 1):
    path = sample["path"]
    fname = sample["file_name"]
    print(f"\n==================== Test Image {i:2d}/10: {fname} ====================")
    if not os.path.exists(path):
        print(f"   [File Missing] Image not found at: {path}")
        continue
        
    # 1. Ground Truth
    print(f"  [Ground Truth] ({len(sample['gt_boxes'])} tags):")
    if sample["gt_boxes"]:
        for b, l in zip(sample["gt_boxes"], sample["gt_labels"]):
            print(f"     * {l:<15} -> Box: {b}")
    else:
        print("     * (No ground truth annotations for this image)")
        
    # 2. Multi-Class RF-DETR
    rf_tags = predict_tags_rfdetr(path)
    print(f"  [Multi-Class RF-DETR] ({len(rf_tags)} tags detected):")
    if rf_tags:
        for t in rf_tags:
            print(f"     * {t['tag']:<15} (conf: {t['score']:.2f}) -> Box: {t['box']}")
    else:
        print("     * No tags detected.")
        
    # 3. Old Model (model.plan) - only run if ready
    if trt_runner.is_ready:
        plan_tags = predict_tags_plan(path)
        print(f"  [Old Model (model.plan)] ({len(plan_tags)} tags detected):")
        if plan_tags:
            for t in plan_tags:
                print(f"     * {t['tag']:<15} (conf: {t['score']:.2f}) -> Box: {t['box']}")
        else:
            print("     * (No tags detected above confidence threshold)")
    else:
        print("  [Old Model (model.plan)]: Skipped (engine not ready / not loaded)")


In [ ]:
# 6. Display Previews for All 10 Images Side-by-Side
GT_COLORS = {
    "blue_aisle": (0, 215, 255),    # Cyan
    "blue_bay": (255, 175, 0),       # Gold
    "location_tag": (0, 255, 150),   # Mint Green
}
PRED_COLORS = {
    "blue_aisle": (30, 144, 255),   # Dodger Blue
    "blue_bay": (255, 120, 0),      # Orange
    "location_tag": (46, 204, 113), # Lime
}

def render_panel(orig_img, boxes, labels, scores=None, is_gt=False, disp_h=520):
    orig_w, orig_h = orig_img.size
    aspect = orig_w / max(orig_h, 1)
    disp_w = max(int(disp_h * aspect), 320)
    
    # Scale image to display resolution BEFORE drawing so lines & badges stay bold & sharp
    disp_img = orig_img.resize((disp_w, disp_h), Image.Resampling.BILINEAR)
    draw = ImageDraw.Draw(disp_img)
    
    sx = disp_w / max(orig_w, 1)
    sy = disp_h / max(orig_h, 1)
    font = ImageFont.load_default()
    
    for idx, (box, label) in enumerate(zip(boxes, labels)):
        dx1 = max(0, min(disp_w - 1, box[0] * sx))
        dy1 = max(0, min(disp_h - 1, box[1] * sy))
        dx2 = max(0, min(disp_w - 1, box[2] * sx))
        dy2 = max(0, min(disp_h - 1, box[3] * sy))
        if dx2 - dx1 < 6: dx2 = min(disp_w - 1, dx1 + 8)
        if dy2 - dy1 < 6: dy2 = min(disp_h - 1, dy1 + 8)
        
        color = GT_COLORS.get(label, (0, 215, 255)) if is_gt else PRED_COLORS.get(label, (235, 40, 40))
        score_str = f" {scores[idx]:.2f}" if (scores and idx < len(scores)) else ""
        badge_text = f"{label}{score_str}" if not is_gt else f"{label} [GT]"
        
        # Bold 4px rectangle outline
        draw.rectangle([dx1, dy1, dx2, dy2], outline=color, width=4)
        
        # Filled label badge
        badge_w = min(max(len(badge_text) * 7 + 8, 70), disp_w - dx1)
        badge_top = max(0, dy1 - 18) if dy1 >= 18 else dy1
        draw.rectangle([dx1, badge_top, dx1 + badge_w, badge_top + 16], fill=color)
        text_fill = (0, 0, 0) if is_gt else (255, 255, 255)
        draw.text((dx1 + 4, badge_top + 2), badge_text, fill=text_fill, font=font)
        
    return disp_img

valid_samples = [s for s in test_samples if os.path.exists(s["path"])]
if valid_samples:
    num_s = len(valid_samples)
    show_old = trt_runner.is_ready
    num_cols = 3 if show_old else 2
    
    fig, axes = plt.subplots(num_s, num_cols, figsize=(6 * num_cols, 4.5 * num_s))
    axes = np.atleast_2d(axes)
        
    for i, s in enumerate(valid_samples):
        path = s["path"]
        base_im = Image.open(path).convert("RGB")
        col_idx = 0
        
        # 1. Panel 1: Ground Truth
        p1 = render_panel(base_im, s["gt_boxes"], s["gt_labels"], is_gt=True)
        axes[i, col_idx].imshow(p1)
        axes[i, col_idx].set_title(f"[Sample {i+1}/10] Ground Truth ({len(s['gt_boxes'])} tags)", fontsize=11, fontweight="bold")
        axes[i, col_idx].axis("off")
        col_idx += 1
        
        # 2. Panel 2: Old model.plan (only run if model is ready)
        if show_old:
            plan_preds = predict_tags_plan(path)
            p_boxes = [p["box"] for p in plan_preds]
            p_labels = [p["tag"] for p in plan_preds]
            p_scores = [p["score"] for p in plan_preds]
            p2 = render_panel(base_im, p_boxes, p_labels, scores=p_scores, is_gt=False)
            axes[i, col_idx].imshow(p2)
            axes[i, col_idx].set_title(f"[Sample {i+1}/10] Old model.plan ({len(plan_preds)} tags)", fontsize=11)
            axes[i, col_idx].axis("off")
            col_idx += 1
        
        # 3. Panel: Multi-Class RF-DETR
        rf_preds = predict_tags_rfdetr(path)
        r_boxes = [p["box"] for p in rf_preds]
        r_labels = [p["tag"] for p in rf_preds]
        r_scores = [p["score"] for p in rf_preds]
        p3 = render_panel(base_im, r_boxes, r_labels, scores=r_scores, is_gt=False)
        axes[i, col_idx].imshow(p3)
        axes[i, col_idx].set_title(f"[Sample {i+1}/10] Multi-Class RF-DETR ({len(rf_preds)} tags)", fontsize=11, fontweight="bold")
        axes[i, col_idx].axis("off")
        
    plt.tight_layout()
    plt.show()
else:
    print("Test image files not found on disk. Check paths in Cell 2.")
